# Notebook 9W — Original A: Comment-Only RoBERTa — Women Discourse

Pure baseline: comment text only → RoBERTa → 3-class distribution.

This version is cloned from the immigration Original A notebook, but configured for **women discourse**. It still gives RoBERTa only the raw comment text; subgroup is used only to choose the subgroup-specific target distribution and for subgroup-level evaluation.

Allowed women subgroups are configurable below. By default, only `women`, `men`, and `non_binary` are kept, so `prefer_not_to_say` and `self_describe` are excluded from training/evaluation.


In [4]:
import ast, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42
MODEL_NAME = "roberta-base"
NUM_LABELS = 3

MAX_LENGTH = 192
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1

EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------
# Women discourse configuration
# ---------------------------------------------------------------------
DISCOURSE = "women"
ALLOWED_SUBGROUPS = ["women", "men", "non_binary"]

# This notebook expects an expanded subgroup-level parquet file with columns:
# comment_id, split, subgroup, text, target_distribution, target_majority_label.
# Update DATA_PATH if your women subgroup-example file lives somewhere else.
DATA_PATH = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts/women_context_mapped_examples.parquet")

# Fallback candidates, useful if your previous notebook saved the file under a different folder/name.
DATA_PATH_CANDIDATES = [
    DATA_PATH,
    Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_model_examples.parquet"),
    Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/simple_token_outputs/women_model_examples.parquet"),
    Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_artifacts/women_context_mapped_examples.parquet"),
]

OUTPUT_DIR = Path("women_original_a_comment_only_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)

print("Device:", DEVICE)
print("Allowed subgroups:", ALLOWED_SUBGROUPS)
print("Output directory:", OUTPUT_DIR.resolve())


Device: cuda
Allowed subgroups: ['women', 'men', 'non_binary']
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_original_a_comment_only_outputs


## 1. Load women subgroup-level data

This loads the expanded women examples, then filters them so only the selected gender subgroups are included. The model input remains **comment text only**.


In [5]:
resolved_path = next((p for p in DATA_PATH_CANDIDATES if p.exists()), None)
if resolved_path is None:
    tried_paths = "\n".join(str(p) for p in DATA_PATH_CANDIDATES)
    raise FileNotFoundError(
        "Could not find the women subgroup-example parquet file. "
        "Update DATA_PATH to the file created by your women expansion notebook. "
        f"Tried:\n{tried_paths}"
    )

print("Using data file:", resolved_path)
df = pd.read_parquet(resolved_path)

print("Raw dataset:", df.shape)
display(df.head(2))

required_columns = ["comment_id", "split", "subgroup", "text", "target_distribution", "target_majority_label"]
missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Normalise possible spelling variants so the selector is reliable.
df["subgroup"] = df["subgroup"].astype(str).str.strip().str.lower().str.replace("-", "_", regex=False)

print("Rows by subgroup before filtering:")
display(df["subgroup"].value_counts())

before = len(df)
df = df[df["subgroup"].isin(ALLOWED_SUBGROUPS)].reset_index(drop=True)
after = len(df)
print(f"Kept {after:,}/{before:,} rows after subgroup filtering.")

if df.empty:
    raise ValueError("No rows remain after filtering. Check ALLOWED_SUBGROUPS and subgroup names in the data.")

print("Rows by split:")
display(df["split"].value_counts())

print("Rows by subgroup after filtering:")
display(df["subgroup"].value_counts())

print("Target majority distribution:")
display(df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Unique comments:", df["text"].nunique())
print("Total subgroup rows:", len(df))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
    raise ValueError("One or more splits are empty after filtering. Check the input data and split labels.")


Using data file: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/women_models/women_context_artifacts/women_context_mapped_examples.parquet
Raw dataset: (7962, 16)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Male privilege],[https://en.wikipedia.org/wiki/Male_privilege],[0.10770449787378311],"[In patriarchal societies, males hold primary power and predominate in roles of leadership, moral authority, social privilege, and control of property, granting them economic, political, social, educational, and prac...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nmen\n\n### RETRIEVED BAC...,35,195
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,[Misogyny],[https://en.wikipedia.org/wiki/Misogyny],[0.07197389751672745],"[Misogyny is a form of sexism that perpetuates women's subordinate status in patriarchal societies. It can manifest in obvious and subtle ways, including violence against women, sexual harassment, and exclusion from ...",### COMMENT TO CLASSIFY\nFirst off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;)\n\n### ANNOTATOR GENDER\nwomen\n\n### RETRIEVED B...,35,189


Rows by subgroup before filtering:


subgroup
women         3919
men           3903
non_binary     140
Name: count, dtype: int64

Kept 7,962/7,962 rows after subgroup filtering.
Rows by split:


...

Rows by subgroup after filtering:


subgroup
women         3919
men           3903
non_binary     140
Name: count, dtype: int64

Target majority distribution:


target_majority_label
0    0.677594
1    0.073851
2    0.248556
Name: proportion, dtype: float64

Unique comments: 3951
Total subgroup rows: 7962
Train: (5565, 16)
Val: (1224, 16)
Test: (1173, 16)


## 2. Tokenizer and token lengths

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text):
    return len(tokenizer(text, truncation=False, add_special_tokens=True)["input_ids"])

df["comment_token_length"] = df["text"].apply(count_tokens)

display(pd.DataFrame([{
    "mean": df["comment_token_length"].mean(),
    "median": df["comment_token_length"].median(),
    "p95": df["comment_token_length"].quantile(0.95),
    "max": df["comment_token_length"].max(),
    "pct_over_192": float((df["comment_token_length"] > 192).mean()),
    "pct_over_256": float((df["comment_token_length"] > 256).mean()),
}]))


,mean,median,p95,max,pct_over_192,pct_over_256
0,29.113665,22.0,74.95,143,0.0,0.0


## 3. Dataset and model

In [7]:
def parse_distribution(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).tolist()
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        return [float(x) for x in ast.literal_eval(value)]
    raise TypeError(f"Unsupported distribution type: {type(value)}")


class CommentOnlyDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "target_distribution": torch.tensor(parse_distribution(row["target_distribution"]), dtype=torch.float),
        }


class CommentOnlyRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


train_loader = DataLoader(CommentOnlyDataset(train_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CommentOnlyDataset(val_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(CommentOnlyDataset(test_df, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items()})

model = CommentOnlyRoBERTa(MODEL_NAME, NUM_LABELS, DROPOUT).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS))
num_training_steps = num_update_steps_per_epoch * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


{'input_ids': torch.Size([16, 192]), 'attention_mask': torch.Size([16, 192]), 'target_distribution': torch.Size([16, 3])}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training steps: 3480
Warmup steps: 348


## 4. Metrics

In [8]:
EPS = 1e-12

def kl_divergence(y_true, y_pred):
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)

def js_divergence(y_true, y_pred):
    return np.array([jensenshannon(t, p, base=2) ** 2 for t, p in zip(y_true, y_pred)])

def cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)

def expected_scores(distributions):
    return distributions @ np.arange(distributions.shape[1])

def entropy_values(distributions):
    return np.array([entropy(d, base=2) for d in distributions])

def compute_metrics(y_true, y_pred):
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)
    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    out = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(expected_scores(y_true), expected_scores(y_pred))),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        out["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        out["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        out["entropy_pearson"] = np.nan
        out["entropy_spearman"] = np.nan

    return out


## 5. Train

In [9]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets, all_predictions = [], []

    if is_training:
        optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(input_ids, attention_mask)
            raw_loss = criterion(outputs["log_probs"], targets)

            if is_training:
                (raw_loss / GRADIENT_ACCUMULATION_STEPS).backward()
                should_step = ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0) or ((step + 1) == len(dataloader))
                if should_step:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    optimizer.zero_grad()
                    if scheduler is not None:
                        scheduler.step()

        total_loss += raw_loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)
    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(total_loss / len(dataloader.dataset))
    return metrics, y_true, y_pred


best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / "women_original_a_comment_only_best_model.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    train_metrics, _, _ = run_epoch(model, train_loader, optimizer, scheduler)
    val_metrics, _, _ = run_epoch(model, val_loader)

    row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print("Saved best model")

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_DIR / "women_original_a_comment_only_training_history.csv", index=False)


Epoch 1/10
Train KL: 0.7340 | Val KL: 0.6359
Train JS: 0.2968 | Val JS: 0.2583
Val Acc: 0.7198 | Val Macro F1: 0.4798
Saved best model
Epoch 2/10
Train KL: 0.6106 | Val KL: 0.6353
Train JS: 0.2365 | Val JS: 0.2313
Val Acc: 0.7165 | Val Macro F1: 0.4758
Saved best model
Epoch 3/10
Train KL: 0.5633 | Val KL: 0.6697
Train JS: 0.2173 | Val JS: 0.2448
Val Acc: 0.7092 | Val Macro F1: 0.4398
Epoch 4/10
Train KL: 0.5309 | Val KL: 0.6634
Train JS: 0.2044 | Val JS: 0.2373
Val Acc: 0.7214 | Val Macro F1: 0.4745
Epoch 5/10
Train KL: 0.5000 | Val KL: 0.6581
Train JS: 0.1946 | Val JS: 0.2353
Val Acc: 0.7214 | Val Macro F1: 0.4798
Epoch 6/10
Train KL: 0.4641 | Val KL: 0.6857
Train JS: 0.1807 | Val JS: 0.2365
Val Acc: 0.7222 | Val Macro F1: 0.4709
Epoch 7/10
Train KL: 0.4310 | Val KL: 0.7179
Train JS: 0.1697 | Val JS: 0.2369
Val Acc: 0.7198 | Val Macro F1: 0.4724
Epoch 8/10
Train KL: 0.4045 | Val KL: 0.7689
Train JS: 0.1598 | Val JS: 0.2339
Val Acc: 0.7206 | Val Macro F1: 0.4743
Epoch 9/10
Train KL: 0

,epoch,train_kl_mean,train_js_mean,train_cross_entropy_mean,train_accuracy,train_macro_f1,train_expected_label_mae,train_entropy_pearson,train_entropy_spearman,train_loss,val_kl_mean,val_js_mean,val_cross_entropy_mean,val_accuracy,val_macro_f1,val_expected_label_mae,val_entropy_pearson,val_entropy_spearman,val_loss
0,1,0.733986,0.296830,0.814592,0.689847,0.402061,0.669749,0.078673,0.061893,0.733986,0.635891,0.258287,0.709021,0.719771,0.479790,0.585412,0.137919,0.133598,0.635891
1,2,0.610552,0.236452,0.691158,0.723270,0.470053,0.539818,0.153780,0.147253,0.610552,0.635289,0.231283,0.708419,0.716503,0.475819,0.510202,0.139222,0.128340,0.635289
2,3,0.563269,0.217268,0.643875,0.739982,0.485814,0.488026,0.177610,0.173111,0.563269,0.669665,0.244769,0.742795,0.709150,0.439836,0.529442,0.148615,0.141131,0.669665
3,4,0.530942,0.204423,0.611549,0.760827,0.500651,0.453985,0.201607,0.191105,0.530943,0.663367,0.237273,0.736497,0.721405,0.474478,0.512030,0.140767,0.132607,0.663367
4,5,0.500035,0.194605,0.580641,0.771608,0.509347,0.429743,0.202904,0.189813,0.500035,0.658073,0.235285,0.731203,0.721405,0.479787,0.511471,0.143555,0.124424,0.658073
5,6,0.464117,0.180657,0.544723,0.782031,0.518062,0.398793,0.218597,0.207038,0.464117,0.685679,0.236461,0.758809,0.722222,0.470902,0.506720,0.140588,0.122300,0.685679
6,7,0.430950,0.169672,0.511557,0.790117,0.545859,0.376714,0.254511,0.241472,0.430951,0.717887,0.236909,0.791017,0.719771,0.472402,0.508150,0.127649,0.112594,0.717887
7,8,0.404492,0.159786,0.485098,0.796586,0.581996,0.360122,0.253963,0.237810,0.404492,0.768934,0.233933,0.842064,0.720588,0.474325,0.497827,0.133547,0.118296,0.768934
8,9,0.390773,0.155236,0.471379,0.796406,0.608802,0.351884,0.269150,0.253937,0.390773,0.774083,0.239034,0.847213,0.713235,0.481934,0.500626,0.129873,0.116307,0.774083
9,10,0.364466,0.146659,0.445072,0.810782,0.657998,0.338772,0.274209,0.259750,0.364466,0.796897,0.235395,0.870027,0.714052,0.481132,0.493797,0.132655,0.118636,0.796897


## 6. Test and diagnostics

In [10]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(model, test_loader)
display(test_metrics)

with open(OUTPUT_DIR / "women_original_a_comment_only_test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

predictions_df = test_df.copy()
predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)
predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)
predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

predictions_df.to_parquet(OUTPUT_DIR / "women_original_a_comment_only_test_predictions.parquet", index=False)
predictions_df.to_csv(OUTPUT_DIR / "women_original_a_comment_only_test_predictions.csv", index=False)

true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())

print("\\nAverage target distribution:")
print(np.vstack(predictions_df["true_distribution"].to_numpy()).mean(axis=0))

print("\\nAverage predicted distribution:")
print(np.vstack(predictions_df["pred_distribution"].to_numpy()).mean(axis=0))

print("\\nMean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_distribution", lambda x: np.mean([entropy(parse_distribution(v), base=2) for v in x])),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)

print("\\nPerformance by subgroup, even though subgroup was not given:")
subgroup_rows = []
for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())
    subgroup_rows.append({"subgroup": subgroup, "n": len(group), **compute_metrics(y_true, y_pred)})

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)
subgroup_metrics_df.to_csv(OUTPUT_DIR / "women_original_a_comment_only_subgroup_metrics.csv", index=False)


/tmp/ipykernel_48675/2655788491.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.672378659248352,
 'js_mean': 0.2419554844740732,
 'cross_entropy_mean': 0.7558417916297913,
 'accuracy': 0.7016197783461211,
 'macro_f1': 0.46000985106585396,
 'expected_label_mae': 0.5517044537911113,
 'entropy_pearson': 0.13830030501330853,
 'entropy_spearman': 0.13710222593798255,
 'loss': 0.6723787057493755}

Confusion matrix:
[[613   0 172]
 [ 52   0  34]
 [ 92   0 210]]
\nClassification report:
              precision    recall  f1-score   support

           0       0.81      0.78      0.80       785
           1       0.00      0.00      0.00        86
           2       0.50      0.70      0.58       302

    accuracy                           0.70      1173
   macro avg       0.44      0.49      0.46      1173
weighted avg       0.67      0.70      0.68      1173

\nPredicted label distribution:


0    0.645354
2    0.354646
Name: proportion, dtype: float64

\nTrue label distribution:


0    0.669224
1    0.073316
2    0.257460
Name: proportion, dtype: float64

\nAverage target distribution:
[0.62618774 0.07862237 0.2951899 ]
\nAverage predicted distribution:
[0.66611314 0.05753761 0.27634928]
\nMean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,785,0.347584,0.141350,0.142549,0.721413
1,86,2.497925,0.746295,0.197674,0.948125
2,302,0.996771,0.359843,0.040868,1.097036


Average predicted distribution by true majority label:
0 [0.7547694  0.05231214 0.192918  ]
1 [0.6184838  0.06547911 0.31603703]
2 [0.4492279  0.06885877 0.48191348]
\nPerformance by subgroup, even though subgroup was not given:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,non_binary,16,0.556892,0.214528,0.556892,0.750000,0.468254,0.501526,NaN,NaN
0,men,576,0.652763,0.236134,0.745357,0.687500,0.440182,0.545157,0.191992,0.184809
2,women,581,0.695006,0.248483,0.771715,0.714286,0.477660,0.559577,0.082064,0.086885


## 7. Same-comment prediction consistency

In [11]:
predictions_df["pred_tuple"] = predictions_df["pred_distribution"].apply(lambda x: tuple(np.round(np.array(x), 6)))

comment_pred_counts = predictions_df.groupby("comment_id")["pred_tuple"].nunique().describe()
display(comment_pred_counts)

multi = predictions_df.groupby("comment_id")["pred_tuple"].nunique().reset_index(name="n_pred_variants")
print("Comments with multiple rounded prediction variants:")
display(multi[multi["n_pred_variants"] > 1].head())


count    582.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: pred_tuple, dtype: float64

Comments with multiple rounded prediction variants:


,comment_id,n_pred_variants


## Interpretation

This is the pure predictive baseline for women discourse:

```text
P(label distribution | comment)
```

The model receives **only the comment text**. It does not receive a gender token, subgroup embedding, ideology token, context text, or FiLM conditioning.

Because the dataset is still subgroup-expanded, the same comment may appear once for `women`, once for `men`, and once for `non_binary`, each with that subgroup's target distribution. Since subgroup identity is not part of the input, this baseline cannot intentionally produce different predictions for different gender groups. Any repeated-comment predictions should therefore be identical apart from numerical/batching noise.
